#  Film Industry Analysis for Business Expansion 

## 🎬 **Introduction**  
As part of a company's expansion into the entertainment industry, they are exploring the creation of a new movie studio. However, with no prior experience in film production, it is critical to make data-driven decisions about the types of movies to develop.  

In this project, I will analyze **movie industry data** to answer key business questions such as:  
- **Which movie genres tend to receive the highest ratings?**  
- **Which genres have the highest return on investment (ROI)?**  
- **Are certain directors more likely to deliver high foreign box office success?**  
- **Which film languages are associated with higher popularity and profitability?**  

The goal is to uncover patterns and insights that will help guide the company's decisions on what types of films to produce for maximum commercial success.


## 【1】**Data Exploration & Cleaning**  

### rt ratings and genre data prep
#### Kevin can make edits to this markdown cell appropriately in your own branch, 
as you guys can see we have mixed both data exploration and data cleaning into one so as you are prepping the data for analysis use appropriate markdown cells where necessary to explain yourself and also do not edit the other cells without your name in it as it will raise merge conflicts when merging all our code together
so you can add as many markdown and code cells in this space before the next guys name.
also if you want to edit anything in the document make note of it and dont change it in your branch just work on the assigned code task in your branch overall fine tuning we will come to that later.
this markdown cell is just as a reminder when you start the work you can remove everything here and make appropriate subtitles and markdown cells for your code

### genres and ROI data prep

As the company explores creating a new movie studio with no prior experience in film production, thorough and correct data preparation is essential to generate actionable insights. Two critical parts of the preparation involve handling genres and calculating Return on Investment (ROI).


**Genre Processing**
Movies often have multiple genres listed (e.g., "action & adventure, science fiction & fantasy, Classics,etc "). For clear analysis:

**Primary Genre Extraction:

Focus on the first listed genre as the primary driver for classification.

This simplifies genre-based aggregation and ROI analysis.

**Standardization:

Normalize genres to ensure consistency (e.g., merge "science fiction & fantasy" under a single label).


**ROI Calculation**
The Return on Investment (ROI) is a key metric for understanding movie profitability.
Definition:

**ROI**

ROI= (Worldwide Gross−Production Budget)/Production Budget * 100

**Steps:

1.Ensure Worldwide Gross and Production Budget columns are numeric and non-null.

2.Handle missing or zero budgets carefully to avoid division errors.

3.Create a new column called ROI.

4.Considering the Genre details is on a different table, join the tables to have all information in one table, for ease in analysis.

5.Final table to have the genre, Movie and Roi in descending order

6.Analsye the data and Make useful Insights

**The Tables To be Used for this purpose is as Below**
-tn.movie_budgets- This will provide information on Production Budget, World Wide Gross as well as Roi feature engineered Column
-Rt_Movies which provides information on Genres
-We will then merge and Make insightful meanings for our objective




In [1]:
# Exploring a code linking tn.movie and rt_movies to have Genre and Roi side by side comparison

# Import necessary libraries
import pandas as pd
import numpy as np
import sqlite3

# Load both datasets
tn_df = pd.read_csv('Data/tn.movie_budgets.csv')
rt_df = pd.read_csv('Data/rt_movies.csv')

# Step 1: Clean column names for easier handling
tn_df.columns = tn_df.columns.str.strip().str.lower().str.replace(' ', '_')
rt_df.columns = rt_df.columns.str.strip().str.lower().str.replace(' ', '_')

# Step 2: Prepare titles for better matching (remove special characters, lower case)
tn_df['movie'] = tn_df['movie'].str.strip().str.lower()
rt_df['movie_title'] = rt_df['movie_title'].str.strip().str.lower()

# Step 3: Merge on movie titles
merged_df = tn_df.merge(rt_df[['movie_title', 'genres']], left_on='movie', right_on='movie_title', how='left')

# Step 4: Drop duplicate movie_title column if you want
merged_df = merged_df.drop(columns=['movie_title'])

# Preview
merged_df.head()


,id,release_date,movie,production_budget,domestic_gross,worldwide_gross,genres
0,1,"Dec 18, 2009",avatar,"$425,000,000","$760,507,625","$2,776,345,279","Action & Adventure, Comedy, Mystery & Suspense..."
1,2,"May 20, 2011",pirates of the caribbean: on stranger tides,"$410,600,000","$241,063,875","$1,045,663,875","Action & Adventure, Comedy, Science Fiction & ..."
2,3,"Jun 7, 2019",dark phoenix,"$350,000,000","$42,762,350","$149,762,350","Action & Adventure, Drama, Science Fiction & F..."
3,4,"May 1, 2015",avengers: age of ultron,"$330,600,000","$459,005,868","$1,403,013,963","Action & Adventure, Science Fiction & Fantasy"
4,5,"Dec 15, 2017",star wars ep. viii: the last jedi,"$317,000,000","$620,181,382","$1,316,721,747",NaN


In [2]:
# The data has been successfully merged using movie column
# lets now proceed to calculate ROI 
# ROI= (worlwide_gross-production_budget)/production _budget in percentage
#Then on the Movie column we group by against the roi
#final table to be a comparison between the genres and the grouped by movie table in that order
#we make meaningful inference from the results
# The code is as below:

# Clean up the dollar signs and commas and convert columns to numeric
merged_df['production_budget'] = merged_df['production_budget'].replace('[\$,]', '', regex=True).astype(float)
merged_df['worldwide_gross'] = merged_df['worldwide_gross'].replace('[\$,]', '', regex=True).astype(float)

# ✅ Calculate ROI = (worldwide_gross - production_budget) / production_budget * 100
merged_df['ROI (%)'] = ((merged_df['worldwide_gross'] - merged_df['production_budget']) / merged_df['production_budget']) * 100

# ✅ Group by 'movie' and calculate the mean ROI (though each movie is unique here)
roi_grouped = merged_df.groupby('movie')[['ROI (%)']].mean().reset_index()

# ✅ Final table: Comparison between genres and the grouped movie ROI
final_table = merged_df[['genres', 'movie']].merge(roi_grouped, on='movie')

# ✅ Display final table
final_table



,genres,movie,ROI (%)
0,"Action & Adventure, Comedy, Mystery & Suspense...",avatar,553.257713
1,"Action & Adventure, Comedy, Science Fiction & ...",pirates of the caribbean: on stranger tides,154.667286
2,"Action & Adventure, Drama, Science Fiction & F...",dark phoenix,-57.210757
3,"Action & Adventure, Science Fiction & Fantasy",avengers: age of ultron,324.384139
4,NaN,star wars ep. viii: the last jedi,315.369636
...,...,...,...
6222,NaN,red 11,-100.000000
6223,"Art House & International, Drama, Mystery & Su...",following,3908.250000
6224,NaN,return to the land of wonders,-73.240000
6225,NaN,a plague so pleasant,-100.000000


In [3]:
# From The table above lets drop the NULL Values so that we have a refined data
# make # ROI= (worlwide_gross-production_budget)/production _budget * 100
#the appropriate code

# Step 1: Drop rows with NULLs
merged_df = merged_df.dropna()

# Step 2: Clean the dollar signs and commas if not already done
merged_df['production_budget'] = merged_df['production_budget'].replace('[\$,]', '', regex=True).astype(float)
merged_df['worldwide_gross'] = merged_df['worldwide_gross'].replace('[\$,]', '', regex=True).astype(float)

# Step 3: Calculate ROI as per formula
merged_df['ROI (%)'] = ((merged_df['worldwide_gross'] - merged_df['production_budget']) / merged_df['production_budget']) * 100

# Step 4: Build the final table with genres, movie, and ROI
final_table = merged_df[['genres', 'movie', 'ROI (%)']]

# Step 5: View the cleaned table
final_table


,genres,movie,ROI (%)
0,"Action & Adventure, Comedy, Mystery & Suspense...",avatar,553.257713
1,"Action & Adventure, Comedy, Science Fiction & ...",pirates of the caribbean: on stranger tides,154.667286
2,"Action & Adventure, Drama, Science Fiction & F...",dark phoenix,-57.210757
3,"Action & Adventure, Science Fiction & Fantasy",avengers: age of ultron,324.384139
6,"Action & Adventure, Science Fiction & Fantasy",avengers: infinity war,582.711400
...,...,...,...
6218,"Action & Adventure, Art House & International,...",el mariachi,29070.400000
6219,"Drama, Mystery & Suspense, Science Fiction & F...",primer,11927.514286
6220,"Art House & International, Horror, Mystery & S...",cavite,923.485714
6223,"Art House & International, Drama, Mystery & Su...",following,3908.250000


In [4]:
# Lets now rank roi from the largest to the smallest
#find the top 5
# code is as below

# Sort the final_table by ROI (%) in descending order
final_table_sorted = final_table.sort_values(by='ROI (%)', ascending=False)

# View the ranked table
final_table_sorted.head()


,genres,movie,ROI (%)
6052,"Action & Adventure, Cult Movies, Science Ficti...",mad max,49775.000000
5926,"Horror, Mystery & Suspense",paranormal activity,43051.785333
6122,"Horror, Mystery & Suspense",the gallows,41556.474000
5832,Horror,the blair witch project,41283.333333
6153,"Documentary, Special Interest",super size me,34105.858462


### language and popularity data prep
#### Chris can make edits to this markdown cell appropriately in your own branch, 
as you guys can see we have mixed both data exploration and data cleaning into one so as you are prepping the data for analysis use appropriate markdown cells where necessary to explain yourself and also do not edit the other cells without your name in it as it will raise merge conflicts when merging all our code together
so you can add as many markdown and code cells in this space before the next guys name.
also if you want to edit anything in the document make note of it and dont change it in your branch just work on the assigned code task in your branch overall fine tuning we will come to that later.
this markdown cell is just as a reminder when you start the work you can remove everything here and make appropriate subtitles and markdown cells for your code

### directors and foreign gross data prep
#### VIhaan can make edits to this markdown cell appropriately in your own branch, 

## 【2】**Data Analysis**  

#### *1. Data Analysis*  

#### *2. Hypothesis Testing*  

## 【3】**Business Recommendations**  

## 【4】**Limitations of Our Analysis**  